In [15]:
import pandas as pd
import os


In [16]:
df=pd.read_csv('/content/sales_raw.csv')
df.head()

,date,product,price,quantity
0,2024-01-01,laptop,1000.0,2.0
1,2024-01-01,laptop,1000.0,2.0
2,2024-01-02,mouse,NaN,5.0
3,2024-01-03,keyboard,50.0,3.0
4,2024-01-04,monitor,200.0,1.0


In [17]:
df=df.drop_duplicates()
df


,date,product,price,quantity
0,2024-01-01,laptop,1000.0,2.0
2,2024-01-02,mouse,NaN,5.0
3,2024-01-03,keyboard,50.0,3.0
4,2024-01-04,monitor,200.0,1.0
5,2024-01-05,mouse,10.0,5.0
7,2024-01-06,keyboard,50.0,NaN
8,2024-01-07,laptop,1000.0,1.0
9,2024-01-08,monitor,200.0,1.0
10,2024-01-09,laptop,1000.0,1.0
11,2024-01-10,mouse,10.0,3.0


In [33]:
df.loc[:, "price"] = df.groupby("product")["price"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.median())
)
df["quantity"] = df["quantity"].fillna(1)

df["price"] = df["price"].astype(float)
df["quantity"] = df["quantity"].astype(int)

df["total"] = df["price"] * df["quantity"]

df["date"] = pd.to_datetime(df["date"])

In [30]:
summary = df.groupby("product").agg(
    total_sales=("total", "sum"),
    total_quantity=("quantity", "sum"),
    avg_price=("price", "mean")
).reset_index()
summary


,product,total_sales,total_quantity,avg_price
0,keyboard,350.0,7,50.0
1,laptop,6000.0,6,1000.0
2,monitor,800.0,4,200.0
3,mouse,190.0,19,10.0


In [34]:

output_file = "output/sales_report.xlsx"

os.makedirs(os.path.dirname(output_file), exist_ok=True)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Cleaned Data", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)

print("Report generated successfully!")
print("Saved to:", output_file)

Report generated successfully!
Saved to: output/sales_report.xlsx
